In [1]:
# 라이브러리 할당
import pandas as pd
import numpy as np
import re

# 데이터셋 불러오기
df_pp = pd.read_csv('ml.csv')

In [2]:
# df 값 보기
print(f"df.shape : {df_pp.shape}")
print("\n[df.info 확인]")
print(df_pp.info())
print(f"\n[df.head 확인] \n {df_pp.head()}")

df.shape : (6929, 28)

[df.info 확인]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6929 entries, 0 to 6928
Data columns (total 28 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   title                6929 non-null   object 
 1   timestamp            6929 non-null   object 
 2   bot                  6929 non-null   bool   
 3   minor                6929 non-null   int64  
 4   length_old           6929 non-null   int64  
 5   length_new           6929 non-null   int64  
 6   len_diff             6929 non-null   int64  
 7   time_delta_sec       6929 non-null   float64
 8   is_first_post        6929 non-null   bool   
 9   type_edit            6929 non-null   bool   
 10  type_log             6929 non-null   bool   
 11  type_new             6929 non-null   bool   
 12  ns_grouped_1         6929 non-null   bool   
 13  ns_grouped_14        6929 non-null   bool   
 14  ns_grouped_2         6929 non-null   bool   
 15  ns

In [5]:
# 데이터 단위 수정 (구조 재정의)
# 1) 필요한 원본 컬럼만 남기기
keep_cols = ["title", "timestamp", "bot", "minor", "len_diff", "time_delta_sec", "is_revert", "is_update", "type_edit", "type_log",
             "type_new", "user_activity_score", "is_pro_editor", "comment_len", "comment_clean_len", "is_section_edit"]

df_rd = df_pp[keep_cols].copy()

# 2) 타입 정리
df_rd["timestamp"] = pd.to_datetime(df_rd["timestamp"], errors="coerce")
df_rd = df_rd.dropna(subset=["title", "timestamp"])

for col in [
    "bot", "minor", "is_revert", "is_update", "type_edit", "type_log", "type_new",
    "is_pro_editor", "is_section_edit"
]:
    df_rd[col] = pd.to_numeric(df_rd[col], errors="coerce").fillna(0).astype(int)

for col in [
    "len_diff", "time_delta_sec", "user_activity_score",
    "comment_len", "comment_clean_len"
]:
    df_rd[col] = pd.to_numeric(df_rd[col], errors="coerce").fillna(0)

In [7]:
# 3) 집계용 파생 컬럼
df_rd["is_human"] = 1 - df_rd["bot"]
df_rd["abs_len_diff"] = df_rd["len_diff"].abs()

# 4) 60초 단위 윈도우
df_rd["window_start"] = df_rd["timestamp"].dt.floor("10min")

In [9]:
# 5) 집계
agg_df = (
    df_rd.groupby(["title", "window_start"], as_index=False)
      .agg(
          edit_count=("title", "size"),
          sum_len_diff=("len_diff", "sum"),
          abs_sum_len_diff=("abs_len_diff", "sum"),
          max_abs_len_diff=("abs_len_diff", "max"),
          human_ratio=("is_human", "mean"),
          minor_ratio=("minor", "mean"),
          revert_ratio=("is_revert", "mean"),
          update_ratio=("is_update", "mean"),
          mean_time_delta_sec=("time_delta_sec", "mean"),
          min_time_delta_sec=("time_delta_sec", "min"),
          mean_user_activity_score=("user_activity_score", "mean"),
          pro_editor_ratio=("is_pro_editor", "mean"),
          section_edit_ratio=("is_section_edit", "mean"),
          mean_comment_len=("comment_len", "mean"),
          mean_comment_clean_len=("comment_clean_len", "mean"),
      )
)

# 6) burst score : 짧은 시간에 편집이 얼마나 몰렸는지(집중도)
# log1p 역수
agg_df["burst_score"] = 1 / np.log1p(agg_df["mean_time_delta_sec"] + 1)

In [11]:
agg_df.head()

,title,window_start,edit_count,sum_len_diff,abs_sum_len_diff,max_abs_len_diff,human_ratio,minor_ratio,revert_ratio,update_ratio,mean_time_delta_sec,min_time_delta_sec,mean_user_activity_score,pro_editor_ratio,section_edit_ratio,mean_comment_len,mean_comment_clean_len,burst_score
0,0,2026-03-16 16:20:00,1,0,0,0,1.0,1.0,0.0,0.0,0.00,0.0,1.0,0.0,1.0,42.0,39.0,1.442695
1,112th Grey Cup,2026-03-16 16:20:00,1,33,33,33,1.0,1.0,0.0,0.0,0.00,0.0,51.0,1.0,0.0,99.0,84.0,1.442695
2,13th Illinois General Assembly,2026-03-16 16:20:00,1,1,1,1,1.0,1.0,0.0,0.0,0.00,0.0,28.0,0.0,1.0,57.0,54.0,1.442695
3,13th Parliament of British Columbia,2026-03-16 16:20:00,1,0,0,0,1.0,1.0,0.0,0.0,0.00,0.0,28.0,0.0,1.0,60.0,57.0,1.442695
4,13th Tripura Assembly,2026-03-16 16:20:00,4,335,335,244,1.0,0.0,0.0,0.0,80.25,0.0,4.0,0.0,1.0,37.0,32.0,0.226770


In [13]:
# KEI 구조

agg_df["quality_score"] = (
    0.6 * (1 - agg_df["revert_ratio"])
    + 0.4 * agg_df["human_ratio"])

agg_df["effective_edit_count"] = (
    agg_df["edit_count"] * (1 - agg_df["minor_ratio"]))


agg_df["KEI"] = (
    np.log1p(agg_df["effective_edit_count"])
    * agg_df["burst_score"]
    * np.log1p(agg_df["abs_sum_len_diff"])
    * agg_df["quality_score"])

In [15]:
agg_df["KEI"].describe()

count    5523.000000
mean        1.419359
std         2.197781
min         0.000000
25%         0.000000
50%         0.000000
75%         2.504827
max        11.236856
Name: KEI, dtype: float64

In [17]:
agg_df.sort_values("KEI", ascending=False).head(20)

,title,window_start,edit_count,sum_len_diff,abs_sum_len_diff,max_abs_len_diff,human_ratio,minor_ratio,revert_ratio,update_ratio,...,min_time_delta_sec,mean_user_activity_score,pro_editor_ratio,section_edit_ratio,mean_comment_len,mean_comment_clean_len,burst_score,quality_score,effective_edit_count,KEI
5220,User:Tabilay/sandbox,2026-03-27 11:10:00,1,75875,75875,75875,1.0,0.0,0.0,0.0,...,0.0,2.0,0.0,0.0,10.0,10.0,1.442695,1.0,1.0,11.236856
1747,Field propulsion,2026-03-16 16:30:00,1,-51051,51051,51051,1.0,0.0,0.0,1.0,...,0.0,8.0,0.0,0.0,235.0,229.0,1.442695,1.0,1.0,10.840600
3485,User:Actuall7/sandbox2,2026-03-27 10:50:00,1,34158,34158,34158,1.0,0.0,0.0,0.0,...,0.0,2.0,0.0,0.0,28.0,28.0,1.442695,1.0,1.0,10.438781
1875,Harka Sampang,2026-03-27 11:00:00,1,-18242,18242,18242,1.0,0.0,0.0,0.0,...,0.0,6.0,0.0,0.0,271.0,247.0,1.442695,1.0,1.0,9.811537
1600,Draft:John V. Heymach,2026-03-16 16:20:00,1,13946,13946,13946,1.0,0.0,0.0,0.0,...,0.0,18.0,0.0,0.0,52.0,48.0,1.442695,1.0,1.0,9.543020
148,2026 Stanley Cup Final,2026-03-16 16:20:00,1,12361,12361,12361,1.0,0.0,0.0,0.0,...,0.0,8.0,0.0,0.0,139.0,139.0,1.442695,1.0,1.0,9.422383
3007,Talk:C. N. R. Rao/Archive 1,2026-03-16 16:30:00,1,11930,11930,11930,1.0,0.0,0.0,0.0,...,0.0,4.0,0.0,0.0,11.0,11.0,1.442695,1.0,1.0,9.386895
1604,Draft:Karting Asia-Pacific Championship,2026-03-27 10:50:00,1,11790,11790,11790,1.0,0.0,0.0,0.0,...,0.0,7.0,0.0,1.0,27.0,22.0,1.442695,1.0,1.0,9.375092
38,1994–95 Orlando Magic season,2026-03-16 16:20:00,1,11066,11066,11066,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,33.0,33.0,1.442695,1.0,1.0,9.311723
2453,Olivier Giacomotto,2026-03-16 16:10:00,1,8535,8535,8535,1.0,0.0,0.0,0.0,...,0.0,9.0,0.0,0.0,136.0,46.0,1.442695,1.0,1.0,9.052048


In [19]:
(agg_df["KEI"] == 0).mean()

0.6061922868006518

## 모델링 1

In [22]:
# 라벨 재정의 : 상위 n%를 트렌드로 설정

threshold = agg_df["KEI"].quantile(0.712)

agg_df["label"] = (agg_df["KEI"] >= threshold).astype(int)

In [24]:
# 모델링
# 1) feature 선택
# features = [ "effective_edit_count", "burst_score", "abs_sum_len_diff", "quality_score", "mean_time_delta_sec",
#             "revert_ratio", "human_ratio"]

features = [
    "mean_time_delta_sec",
    "revert_ratio",
    "human_ratio",
    "mean_user_activity_score",
    "section_edit_ratio"
]

In [26]:
# 2) 데이터 분리
from sklearn.model_selection import train_test_split

X = agg_df[features]
y = agg_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [27]:
# 3) Logistic Regression (baseline) -> feature 의미 있는지 확인용
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.95      0.63      0.76       787
           1       0.50      0.92      0.65       318

    accuracy                           0.71      1105
   macro avg       0.72      0.77      0.70      1105
weighted avg       0.82      0.71      0.72      1105

ROC-AUC: 0.8315672124859149


In [30]:
# 4) Random Forest (2차 baseline) -> 비선형 패턴 확인
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.95      0.76      0.84       787
           1       0.60      0.90      0.72       318

    accuracy                           0.80      1105
   macro avg       0.78      0.83      0.78      1105
weighted avg       0.85      0.80      0.81      1105

ROC-AUC: 0.8895235469460494


In [38]:
import pickle
import os

# RF 모델 저장 
with open('WikiTrend_RF_Model.pkl', 'wb') as f:
    pickle.dump(rf, f)


In [95]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, y_proba_xgb)

# F1 최대화 threshold 찾기
f1_scores = 2 * precision * recall / (precision + recall + 1e-9)
best_threshold = thresholds[f1_scores.argmax()]
print(f"Best threshold: {best_threshold:.3f}, F1: {f1_scores.max():.3f}")

Best threshold: 0.712, F1: 0.489


In [81]:
for t in [0.5, 0.3, 0.2, 0.1]:
    y_pred = (y_prob > t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_test, y_pred))


Threshold: 0.5
              precision    recall  f1-score   support

           0       0.98      0.65      0.78       431
           1       0.22      0.90      0.35        48

    accuracy                           0.67       479
   macro avg       0.60      0.77      0.57       479
weighted avg       0.91      0.67      0.74       479


Threshold: 0.3
              precision    recall  f1-score   support

           0       0.98      0.56      0.71       431
           1       0.19      0.92      0.31        48

    accuracy                           0.59       479
   macro avg       0.59      0.74      0.51       479
weighted avg       0.90      0.59      0.67       479


Threshold: 0.2
              precision    recall  f1-score   support

           0       0.98      0.54      0.70       431
           1       0.18      0.92      0.30        48

    accuracy                           0.58       479
   macro avg       0.58      0.73      0.50       479
weighted avg       0.90   

In [101]:
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# 클래스 비율 계산
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

# 모델 정의
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    tree_method='hist'
)

# 학습
xgb_model.fit(X_train, y_train)

# 확률 / 예측
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_pred_xgb = (y_proba_xgb >= 0.5).astype(int)

# 평가
cm = confusion_matrix(y_test, y_pred_xgb)
print(f'confusion_matrix\n{cm}')
print(classification_report(y_test, y_pred_xgb))
print('ROC-AUC:', roc_auc_score(y_test, y_proba_xgb))

confusion_matrix
[[213 122]
 [ 20 124]]
              precision    recall  f1-score   support

           0       0.91      0.64      0.75       335
           1       0.50      0.86      0.64       144

    accuracy                           0.70       479
   macro avg       0.71      0.75      0.69       479
weighted avg       0.79      0.70      0.72       479

ROC-AUC: 0.8434701492537313


## 모델링 2

In [86]:
threshold_edit = agg_df["edit_count"].quantile(0.9)
agg_df["label"] = (agg_df["edit_count"] >= threshold_edit).astype(int)

In [85]:
# feature 추출
features = [
    "mean_time_delta_sec",
    "revert_ratio",        # quality_score 구성 → 주의
    "human_ratio",         # quality_score 구성 → 주의
    "mean_user_activity_score",
    "section_edit_ratio",
    # ↓ 추가 후보 (KEI 구성 변수이지만 label과의 관계가 간접적)
    "edit_count",          # 편집 수
    "abs_sum_len_diff",    # 콘텐츠 변화량
    "minor_ratio",         # 소규모 편집 비율
    "update_ratio",
    "max_abs_len_diff",
]

In [87]:
# 2) 데이터 분리
from sklearn.model_selection import train_test_split

X = agg_df[features]
y = agg_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [89]:
# 3) 모델링
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight={0: 1, 1: 3}  # 9 대신 3으로 줄임
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00        79

    accuracy                           1.00       479
   macro avg       1.00      1.00      1.00       479
weighted avg       1.00      1.00      1.00       479

ROC-AUC: 1.0
